# BM25 and Hybrid Retrieval — deep dive

This leaf goes deeper than `01-rag/03-hybrid-search/`, covering:

1. **BM25 internals** — IDF, TF saturation, field-length normalisation.
2. **TF-IDF** baseline and why BM25 outperforms it.
3. **RRF** (Reciprocal Rank Fusion) — parameter-free combiner.
4. **Weighted linear fusion** — `α·dense + (1-α)·BM25` with an α sweep.
5. **When BM25 helps, when it hurts** — entity-heavy vs paraphrase queries.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
import re, math
import numpy as np
from rank_bm25 import BM25Okapi
from shared.embedders import cosine_topk, hash_embed
from shared.loaders import load_corpus, load_golden_qa

DOCS = load_corpus()
QA   = [q for q in load_golden_qa() if q.source_ids]
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids   = [d.arxiv_id for d in DOCS]
DIMS = 256
doc_vecs = hash_embed(doc_texts, dims=DIMS, seed=0)

TOKEN_RE = re.compile(r'[A-Za-z0-9]+')
def tokenize(t): return [w.lower() for w in TOKEN_RE.findall(t)]
tokenized = [tokenize(t) for t in doc_texts]

## BM25 formula refresher

$$\text{score}(D,Q) = \sum_{t \in Q} \text{IDF}(t) \cdot \frac{f(t,D) \cdot (k_1+1)}{f(t,D) + k_1(1 - b + b \cdot |D|/\text{avgdl})}$$

Key parameters:
- `k1=1.5` — TF saturation: how quickly additional term repetitions add value
- `b=0.75` — length normalisation: penalise long documents

TF-IDF is `k1=∞, b=0` — no saturation, no length norm.

In [3]:
# Manual BM25 implementation to inspect term-level scores
def idf(term, corpus_tokens, N):
    df = sum(1 for doc in corpus_tokens if term in doc)
    return math.log((N - df + 0.5) / (df + 0.5) + 1)

N = len(tokenized)
avgdl = sum(len(d) for d in tokenized) / N
k1, b = 1.5, 0.75

q_tokens = tokenize('routing aware mixture of experts inference latency')
for t in q_tokens:
    idf_v = idf(t, tokenized, N)
    print(f'  {t:<25}  IDF={idf_v:.3f}')

  routing                    IDF=1.992
  aware                      IDF=1.992
  mixture                    IDF=1.992
  of                         IDF=0.258
  experts                    IDF=1.992
  inference                  IDF=0.526
  latency                    IDF=0.894


## BM25 vs TF-IDF recall

In [4]:
bm25 = BM25Okapi(tokenized, k1=1.5, b=0.75)
bm25_flat = BM25Okapi(tokenized, k1=100.0, b=0.0)  # ≈ TF-IDF

def bm25_recall(idx, k=3):
    hits = 0
    for item in QA:
        scores = idx.get_scores(tokenize(item.question))
        top_k = set(np.argsort(-scores)[:k])
        got = {doc_ids[i] for i in top_k}
        if got & set(item.source_ids): hits += 1
    return round(hits / len(QA), 4)

def dense_recall(k=3):
    hits = 0
    for item in QA:
        qv = hash_embed([item.question], dims=DIMS, seed=0)[0]
        idx_, _ = cosine_topk(qv, doc_vecs, k=k)
        if {doc_ids[i] for i in idx_} & set(item.source_ids): hits += 1
    return round(hits / len(QA), 4)

print(f'TF-IDF recall@3  : {bm25_recall(bm25_flat)}')
print(f'BM25 recall@3    : {bm25_recall(bm25)}')
print(f'Dense recall@3   : {dense_recall()}')

TF-IDF recall@3  : 1.0
BM25 recall@3    : 1.0
Dense recall@3   : 0.8846


## RRF — Reciprocal Rank Fusion

`score(d) = Σ 1/(k + rank_i(d))`  — parameter-free, robust to different score scales.

In [5]:
def rrf_recall(k_rrf=60, k=3):
    hits = 0
    for item in QA:
        qv = hash_embed([item.question], dims=DIMS, seed=0)[0]
        dense_order = list(cosine_topk(qv, doc_vecs, k=len(doc_ids))[0])
        bm25_scores = bm25.get_scores(tokenize(item.question))
        sparse_order = list(np.argsort(-bm25_scores))
        fused = {}
        for rank, idx_ in enumerate(dense_order):
            fused[idx_] = fused.get(idx_, 0) + 1/(k_rrf + rank)
        for rank, idx_ in enumerate(sparse_order):
            fused[idx_] = fused.get(idx_, 0) + 1/(k_rrf + rank)
        top_k = {doc_ids[i] for i in sorted(fused, key=lambda x: -fused[x])[:k]}
        if top_k & set(item.source_ids): hits += 1
    return round(hits / len(QA), 4)

print(f'RRF recall@3: {rrf_recall()}')

RRF recall@3: 0.9615


## Weighted linear fusion — α sweep

`score = α·dense_score + (1-α)·bm25_norm_score`

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def weighted_recall(alpha, k=3):
    hits = 0
    for item in QA:
        qv = hash_embed([item.question], dims=DIMS, seed=0)[0]
        dense_scores = doc_vecs @ qv
        # Normalize dense to [0,1]
        d_min, d_max = dense_scores.min(), dense_scores.max()
        d_norm = (dense_scores - d_min) / (d_max - d_min + 1e-9)
        bm25_s = np.array(bm25.get_scores(tokenize(item.question)), dtype=float)
        b_min, b_max = bm25_s.min(), bm25_s.max()
        b_norm = (bm25_s - b_min) / (b_max - b_min + 1e-9)
        combined = alpha * d_norm + (1 - alpha) * b_norm
        top_k = {doc_ids[i] for i in np.argsort(-combined)[:k]}
        if top_k & set(item.source_ids): hits += 1
    return round(hits / len(QA), 4)

alphas = np.linspace(0, 1, 21)
recalls = [weighted_recall(a) for a in alphas]
fig, ax = plt.subplots(figsize=(5, 2.8))
ax.plot(alphas, recalls, 'o-', markersize=4)
ax.set_xlabel('alpha (1.0 = dense only)')
ax.set_ylabel('recall@3')
ax.set_title('Weighted linear fusion recall vs alpha')
ax.grid(True, alpha=0.3)
fig.tight_layout()
from shared.paths import repo_root
fig.savefig(str(repo_root() / '02-indexing/03-bm25-and-hybrid/fusion_sweep.png'), dpi=120)
plt.close(fig)
best_alpha = alphas[int(np.argmax(recalls))]
print(f'Best alpha={best_alpha:.2f}  recall@3={max(recalls)}')

Best alpha=0.00  recall@3=1.0


## Takeaways

* **BM25 wins on entity queries** (exact term match, e.g. `RA-MoE`, `synth-001`).
* **Dense wins on paraphrase queries** (semantic similarity, e.g. "inference acceleration for sparse models").
* **RRF is robust** — no hyperparameter to tune. Use as default hybrid combiner.
* **Weighted fusion** can squeeze out extra points, but requires a labelled dev set to tune `α`.
* **Rule of thumb:** use RRF for the first prototype; switch to weighted fusion once you have annotations.